In [147]:
import pandas as pd
orders = pd.read_csv('../data/orders.csv').head()
orders

,order_id,customer_id,product_id,qty,date
0,1,1,104,1,2025-07-07
1,1,1,94,2,2025-12-04
2,1,1,389,2,2024-07-17
3,2,1,958,2,2025-08-07
4,2,1,334,2,2025-06-08


In [148]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity



class OrderRecommendationEngine():

    def preprocess(
        self,
        df: pd.DataFrame,
    ) -> pd.DataFrame:
        
        basket = (
            df
            .assign(bought=1)
            .pivot_table(
                index="order_id",
                columns="product_id",
                values="bought",
                aggfunc="max",
                fill_value=0,
            )
            .astype(np.float32)
        )
        return basket

    def train(
        self,
        data: pd.DataFrame,
    ):

        basket = self.preprocess(data)

        self.popular_products = (
            data
            .groupby("product_id")["order_id"]
            .count()
            .sort_values(ascending=False)
        )

        self.similarity_df = pd.DataFrame(
            cosine_similarity(basket.T),
            index=basket.columns,
            columns=basket.columns,
        ).where(
            ~np.eye(basket.shape[1], dtype=bool),
            other=0.0,
        )

        return self

    def predict(
        self,
        sample: pd.DataFrame,
        limit: int = 6,
    ) -> pd.Series:

        cart_ids = set(sample["product_id"].tolist())
        in_matrix = [pid for pid in cart_ids if pid in self.similarity_df.columns]

        if not in_matrix:
            return self.popular_products.head(limit)

        scores = (
            self.similarity_df[in_matrix]
            .sum(axis=1)
            .drop(index=list(cart_ids & set(self.similarity_df.index)))
            .nlargest(limit)
            .replace(0, np.nan).dropna()
        )

        scores = pd.concat(
            [
                scores,
                self.popular_products.drop(index=scores.index).head(limit - len(scores)),
            ],
        ).head(limit)

        return scores


In [149]:
OrderRecommendationEngine().preprocess(orders)

product_id,94,104,334,389,958
order_id,,,,,
1,1.0,1.0,0.0,1.0,0.0
2,0.0,0.0,1.0,0.0,1.0


In [150]:
engine = OrderRecommendationEngine().train(orders)

In [151]:
order = pd.DataFrame(
    [
        {
            'order_id': 1,
            'customer_id': 1,
            'product_id': 94,
            'qty': 2,
            'date': '2025-12-04',
        },
        {
            'order_id': 1,
            'customer_id': 1,
            'product_id': 389,
            'qty': 2,
            'date': '2024-07-17',
        },
    ]
)

order

,order_id,customer_id,product_id,qty,date
0,1,1,94,2,2025-12-04
1,1,1,389,2,2024-07-17


In [152]:
(engine.similarity_df[[94, 389]]
    .sum(axis=1)
    .drop(index=list(set([94, 389]) & set(engine.similarity_df.index)))
    .nlargest(5)
        )

product_id
104    2.0
334    0.0
958    0.0
dtype: float32

In [153]:
engine.predict(order)

product_id
104    2.0
94     1.0
334    1.0
389    1.0
958    1.0
dtype: float64